connectivity measruements across patients
DI, probabilities etc

In [1]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
import pandas as pd

import sys

sys.path.append('T:\EL_experiment\Codes\CCEP_human\Python_Analysis/py_functions')

from scipy.stats import norm
from tkinter import *
import scipy
from scipy import signal

import platform
from glob import glob
from scipy.spatial import distance
import basic_func as bf
from scipy.integrate import simps
from numpy import trapz
import tqdm
from matplotlib.patches import Rectangle
from pathlib import Path
import statsmodels.api as sm
import statsmodels.formula.api as smf

#
from scipy.signal import hilbert, butter, filtfilt
import scipy.stats as stats
from tqdm.notebook import trange, tqdm
cwd             = os.getcwd()

##all 
cond_vals   = np.arange(4)
cond_labels = ['BM', 'BL', 'Fuma', 'Benzo']
cond_colors = ['#494159','#594157', "#F1BF98","#8FB996"]
dist_groups = np.array([[0,15],[15,30],[30,5000]])
dist_labels = ['local (<15 mm)', 'short (<30mm)', 'long']
sub_path  ='X:\\4 e-Lab\\' # y:\\eLab

sys.path.append('T:\\EL_experiment\Codes\CCEP_human\Python_Analysis\py_functions')
sys.path.append('X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram')
sys.path.append('T:\\EL_experiment\Codes\\UBELIX_EvM\\NMF\\functions')
path_connectogram = 'X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram'
from main_script import ConnectogramPlotter
cwp = os.getcwd()

In [3]:
CIRC_AREAS_FILEPATH = 'X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram\circ_areas.xlsx'
tab_region = pd.read_excel(CIRC_AREAS_FILEPATH, sheet_name='plot')
tab_region = tab_region.sort_values('Order').reset_index(drop=True)
regions_all = tab_region.Area.values
region_col = tab_region.color.values

CIRC_AREAS_FILEPATH = 'X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram\circ_areas.xlsx'
all_region = pd.read_excel(CIRC_AREAS_FILEPATH, sheet_name='atlas')

In [4]:
plt.rcParams.update({
            'font.family': 'arial',
            'font.size': 12,
            'xtick.labelsize': 8,
            'ytick.labelsize': 8,
            'legend.fontsize': 9,
            'svg.fonttype': 'none',
            'font.size': 10,
            'axes.titlesize': 10,
            'axes.labelsize': 8,
            'xtick.labelsize': 8,
            'ytick.labelsize': 8,
            'legend.fontsize': 9,
            'figure.titlesize': 10
        })

In [5]:
color_elab = np.zeros((3, 3))
color_elab[0, :] = np.array([31, 78, 121]) / 255
color_elab[1, :] = np.array([189, 215, 238]) / 255
color_elab[2, :] = np.array([0.256, 0.574, 0.431])
DI_color = 'PRGn'

In [6]:
import load_summary as ls
#import BM_across_plots as plotting

cwd = os.getcwd()

##all
cond_vals = np.arange(4)
cond_labels = ['BM', 'BL', 'Fuma', 'Benzo']
cond_colors = ['#494159', '#594157', "#F1BF98", "#8FB996"]
dist_groups = np.array([[0, 15], [15, 30], [30, 5000]])
dist_labels = ['local (<15 mm)', 'short (<30mm)', 'long']
sub_path = 'X:\\4 e-Lab\\'  # y:\\eLab

##
subjs = ["EL010", "EL011","EL012","EL013", "EL014", "EL015", "EL016", "EL017", "EL019", "EL020", "EL021",
         "EL022", "EL026", "EL027", "EL028"]



In [7]:
con_all = ls.get_connections(subjs, sub_path)
con_all_ss = ls.get_connections_sleep(subjs, sub_path, ['Wake'])

FileNotFoundError: [Errno 2] No such file or directory: 'X:\\4 e-Lab\\EvM\\Projects\\EL_experiment\\Analysis\\Patients\\EL011\\BrainMapping\\CR\\data\\summ_W.csv'

In [ ]:
con_all['Con_ID'] = con_all.groupby(['Subj', 'Stim', 'Chan']).ngroup()

In [ ]:
con_all.insert(6, 'Group', 'indirect')
con_all.loc[(con_all.d < 20) & (con_all.delay < 0.02), 'Group'] = 'local direct'
con_all.loc[(con_all.d > 20) & (con_all.delay < 0.02), 'Group'] = 'long direct'

In [ ]:
con_all = con_all[(con_all.Num_sig_trial >10)&(con_all.True_peak == 1)].reset_index(drop=True)
con_all['Con_ID'] = con_all.groupby(['Subj', 'Stim', 'Chan']).ngroup()

In [ ]:
# con_all_ss = ls.get_connections_sleep(subjs, sub_path, ss)
con_all_ss = con_all_ss.merge(con_all[['Subj', 'Stim', 'Chan', 'Con_ID', 'Group']], on=['Subj', 'Stim', 'Chan'])

In [ ]:
stop

In [ ]:
x_P = np.linspace(0, 1, 101)
y_P = np.linspace(0, 1, 101)

# Calculate the matrices
DI_D = np.zeros((len(y_P), len(x_P)))
DI_R = np.zeros((len(y_P), len(x_P)))
DI_C = np.zeros((len(y_P), len(x_P)))

for i in range(len(y_P)):
    for j in range(len(x_P)):
        DI_D[i, j] = x_P[j] - y_P[i]
        min_val = min(x_P[j], y_P[i])
        max_val = max(x_P[j], y_P[i])
        if x_P[j] < y_P[i]:
            DI_R[i, j] = -1 * (1 - min_val / max_val)
        else:
            DI_R[i, j] = 1 - min_val / max_val
        DI_C[i, j] = DI_D[i, j] * (1 - min_val / max_val)
# Create subplots
fig, axs = plt.subplots(1, 3, figsize=(12, 4))

# Plot DI_D
cax1 = axs[0].matshow(DI_D, cmap='PRGn', origin='lower')
axs[0].set_title('DI_D')

# Plot DI_R
cax2 = axs[1].matshow(DI_R, cmap='PRGn', origin='lower')
axs[1].set_title('DI_R')

# Plot DI_C
cax3 = axs[2].matshow(DI_C, cmap='PRGn', origin='lower')
axs[2].set_title('DI_C')

# Create a common colorbar to the right
#cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
#cbar = fig.colorbar(cax3, cax=cbar_ax)

plt.tight_layout()
plt.savefig('X:\\4 e-Lab\EvM\Projects\EL_experiment\Analysis\General_figures\\DI_cal.png', dpi = 300)
plt.show()

In [ ]:
stop